# 04 — Shrinkage and the trees: ν, depth, and the number of trees

*Chapter 08 · Gradient Boosting · Notebook 4 of 6*

Notebooks 1–3 built the engine: gradient boosting fits the negative gradient of whatever loss we choose,
one shrunken tree at a time. This notebook is about **control** — the three dials that decide how well
that engine generalizes:

- **ν**, the learning rate (how big a step each tree takes),
- the tree **depth** (how expressive each step is),
- and **n_estimators**, the number of trees.

And it carries one headline that breaks sharply with Chapter 06: unlike a random forest, **adding more
trees can make gradient boosting worse — at a large learning rate**.

**Prerequisites.** Notebook 1 (the residual loop, F ← F + ν·tree, a regression tree's leaf = the mean);
Notebook 2 (the residual is the negative gradient, ν is the step size); Chapter 06 Notebook 4 (a random
forest: more trees never hurts, and its trees grow deep); Chapter 07 Notebook 3 (how boosting overfits).

**What you'll be able to do.**
- Read a staged test curve and tell under-fitting from over-fitting.
- Set ν against n_estimators knowingly (small steps, more trees).
- Choose tree depth as an **interaction order**.
- Explain *mechanistically* why more trees can hurt gradient boosting while they never hurt a forest.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# Friedman's regression benchmark: a known formula, so we can watch each dial work.
# 5 informative features (including a real x0*x1 interaction) + 5 pure-noise features.
X_raw, y = make_friedman1(n_samples=2000, noise=1.0, random_state=SEED)
X = pd.DataFrame(X_raw, columns=[f"x{i}" for i in range(X_raw.shape[1])])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)

print(f"train {X_train.shape[0]} rows, test {X_test.shape[0]};  "
      f"{X.shape[1]} features (5 informative + 5 noise)")
print(X_train.head(3).round(3))
print(pd.Series(y_train, name="y").describe().round(2))

## A recap, and the data we'll use

The loop from Notebooks 1–2, in one breath: start at the best constant, F₀ = mean(y); each round fit a
regression tree to the residual (which is the negative gradient of the squared-error loss); add a
**shrunken** slice, F ← F + ν·tree; repeat. Three knobs govern how this generalizes — **ν** (the step
size), **depth** (how expressive each tree is), and **n_estimators** (how many steps we take).

To watch each knob work, we use a dataset whose structure we *know*: Friedman's regression benchmark,

> y = 10·sin(π·x₀·x₁) + 20·(x₂ − 0.5)² + 10·x₃ + 5·x₄ + noise.

Ten features are drawn uniformly in [0, 1], but only **five matter** (x₀…x₄) — the other five are pure
noise the model must learn to ignore. Notice the **x₀·x₁ term**: a genuine two-feature *interaction*. We
will use it to see exactly what tree depth buys.

## ν is a step-size brake

Every tree is multiplied by ν before it is added: F ← F + ν·tree. With ν = 1 we take the whole step the
tree proposes; with ν = 0.1 we take a tenth of it and leave the rest for later trees to refine.

Small steps move slowly but can be aimed finely; big steps move fast but tend to overshoot, committing
hard to whatever the current residual says. So ν trades **speed for precision** — and it is tied to the
number of trees: shrink the step, and you need more of them. Let us feel that by hand.

In [ ]:
# F0 is the best constant prediction: the mean of the training target.
F0 = y_train.mean()
base_mse = mean_squared_error(y_train, np.full(len(y_train), F0))
print(f"F0 = mean(y_train) = {F0:.2f}   (train MSE at F0 = {base_mse:.2f})")

# One depth-3 tree fit to the residual, then added at two learning rates.
first_tree = DecisionTreeRegressor(max_depth=3, random_state=SEED).fit(X_train, y_train - F0)
step = first_tree.predict(X_train)
for nu in (1.0, 0.1):
    mse_after = mean_squared_error(y_train, F0 + nu * step)
    print(f"  after 1 tree at nu={nu}:  train MSE {base_mse:.2f} -> {mse_after:.2f}")


# How many trees to reach a fixed train-MSE target, by hand, at each learning rate.
def trees_to_reach(target, nu, max_trees=2000):
    F = np.full(len(y_train), F0)
    for t in range(1, max_trees + 1):
        leaf = DecisionTreeRegressor(max_depth=3, random_state=SEED).fit(X_train, y_train - F)
        F = F + nu * leaf.predict(X_train)
        if mean_squared_error(y_train, F) <= target:
            return t
    return None


for nu in (1.0, 0.5, 0.1, 0.01):
    print(f"  trees to reach train MSE <= 2.0 at nu={nu}: {trees_to_reach(2.0, nu)}")

**Read the result.** F₀ sits at the target's mean. The very first tree, taken at full strength
(ν = 1), drops the training error a long way; the same tree at ν = 0.1 moves only about a tenth as far —
so it takes roughly ten times as many trees to cover the same ground. That is the **ν × n_estimators
trade-off**, here in plain numbers: reaching a training MSE of 2 takes about 13 trees at ν = 1, but ~48 at
ν = 0.1 and ~496 at ν = 0.01.

(One honest nuance: at a large ν the step size is no longer the bottleneck — ν = 1 and ν = 0.5 both arrive
in a dozen-odd trees — so the trade-off really bites as ν gets *small*.) Why pay the price of more trees
at all? Because *where the model ends up* differs — that is the next figure.

## Why small steps generalize better

A full-strength tree chases the training residual hard, committing to whatever it currently says —
including its noise. Small steps instead accumulate many gentle corrections, so the ensemble leans on the
signal that is consistent across rounds and is swayed less by any single tree. In other words,
**shrinkage is a regularizer**. It is not free — it costs trees — but in return it lowers the test error
and guards against over-fitting. The next figure shows both effects at once.

In [ ]:
# Staged test R^2 as trees accumulate, for three learning rates (depth 3).
N_MAX = 1000
trees_axis = np.arange(1, N_MAX + 1)
nu_colors = {1.0: COLORS["error"], 0.1: COLORS["highlight"], 0.01: COLORS["muted"]}

fig, ax = plt.subplots(figsize=(8, 5))
for nu in (1.0, 0.1, 0.01):
    gbr = GradientBoostingRegressor(
        n_estimators=N_MAX, learning_rate=nu, max_depth=3, random_state=SEED
    ).fit(X_train, y_train)
    test_r2 = np.array([r2_score(y_test, p) for p in gbr.staged_predict(X_test)])
    best = int(np.argmax(test_r2))
    ax.plot(trees_axis, test_r2, color=nu_colors[nu], linewidth=2, label=f"ν = {nu}")
    ax.plot(best + 1, test_r2[best], "o", color=nu_colors[nu], markersize=8)
    print(f"nu={nu:<5}: best test R2 = {test_r2[best]:.4f} @ {best + 1} trees,  "
          f"R2 @ {N_MAX} = {test_r2[-1]:.4f}")
ax.set_xscale("log")
ax.set_xlabel("number of trees")
ax.set_ylabel("test R²")
ax.set_title("learning rate × number of trees: small steps reach higher, and stay")
ax.legend()
plt.show()

**Read the figure.** Three learning rates, the same depth-3 budget, test R² as trees accumulate
(note the log scale on the number of trees).

- **ν = 1** (coral) shoots up and peaks — in test R² — after only ~18 trees (R² ≈ 0.864), then **sags to ≈ 0.813** —
  past that point every extra full-strength tree is fitting noise.
- **ν = 0.1** (pink) climbs more slowly to a **higher** peak (≈ 0.930 near 300 trees) and then stays flat
  — more trees do not hurt it within our budget.
- **ν = 0.01** (grey) is **still climbing** at 1000 trees (≈ 0.921): too timid to arrive in time.

So the smaller step reaches a **better** place *and* is **safer** — at the cost of more trees. That is
shrinkage: you trade trees for generalization.

## The headline: more trees can make gradient boosting worse

This is the sharp break from Chapter 06. A random forest **averages independent trees**; adding more only
sharpens the average, so its test error flattens out and never climbs (Chapter 06, Notebook 4). Gradient
boosting is **sequential**: each tree fits the residual the ensemble *currently* leaves, so each one
**adds capacity**. Once the residual is mostly noise, new trees start fitting that noise, and the test
error **turns around and rises**.

"More trees is always safe" is true for a forest and **false for gradient boosting at a large ν**. Let us
see the turnaround, and put a forest beside it for contrast.

In [ ]:
# At nu=1 (no brake), follow train and test MSE as trees accumulate.
gbr1 = GradientBoostingRegressor(
    n_estimators=N_MAX, learning_rate=1.0, max_depth=3, random_state=SEED
).fit(X_train, y_train)
train_mse = np.array([mean_squared_error(y_train, p) for p in gbr1.staged_predict(X_train)])
test_mse = np.array([mean_squared_error(y_test, p) for p in gbr1.staged_predict(X_test)])
bottom = int(np.argmin(test_mse))

# A random forest, for contrast: its test error barely moves as the number of trees B grows.
for B in (50, 200, 1000):
    rf = RandomForestRegressor(n_estimators=B, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
    print(f"random forest B={B:4d}:  test R2 = {rf.score(X_test, y_test):.4f}")
rf_test_mse = mean_squared_error(y_test, rf.predict(X_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(trees_axis, train_mse, color=COLORS["train"], linewidth=2, label="training MSE")
ax.plot(trees_axis, test_mse, color=COLORS["test"], linewidth=2, label="test MSE")
ax.plot(bottom + 1, test_mse[bottom], "o", color=COLORS["error"], markersize=8,
        label=f"test MSE bottoms at {bottom + 1} trees")
ax.axhline(rf_test_mse, color=COLORS["muted"], linestyle=":", linewidth=1.6,
           label="random forest test MSE (≈ flat in B)")
ax.set_xscale("log")
ax.set_xlabel("number of trees")
ax.set_ylabel("MSE")
ax.set_title("at ν=1, more trees overfit gradient boosting; a forest is unmoved")
ax.legend()
print(f"test MSE bottoms {test_mse[bottom]:.2f} @ {bottom + 1} trees, rises to "
      f"{test_mse[-1]:.2f} @ {N_MAX};  train MSE @ {N_MAX} = {train_mse[-1]:.4f}")
plt.show()

**Read the figure.** The training MSE (periwinkle) marches toward zero — with full-strength trees
the ensemble can memorize all 1400 training points. But the test MSE (amber) tells the real story: it
bottoms out at ~18 trees and then **climbs** the rest of the way. The over-fitting is real and, at ν = 1,
fast.

The dotted line is a random forest's test MSE; it barely moves whether the forest has 50 trees or 1000
(test R² ≈ 0.86 throughout). Adding trees to a forest cannot raise its complexity — each tree is an
independent variance-reduction draw (Chapter 06, Notebook 4) — so it cannot over-fit this way. Same words,
"more trees", opposite outcomes: boosting builds **dependence** between its trees, a forest builds
**independence**. This rising curve is exactly the problem Notebook 5 cures with **early stopping**.

## Depth is the interaction order

The third dial is how deep each tree may grow. A **stump** (depth 1) makes a single split, on one feature
— so an ensemble of stumps can only *add up* one-feature effects: it is an additive model, blind to any
interaction between features. Allow **depth 2** and a single leaf can depend on **two** features at once
(say, split on x₀ and then on x₁), which is what it takes to represent a term like x₀·x₁. In general, depth d lets
a tree express interactions of up to d features.

Friedman's formula contains a real two-feature interaction, sin(π·x₀·x₁). So we expect stumps to fall
short here, and depth ≥ 2 to capture what they miss.

In [ ]:
# Depth sweep at a fixed, generous budget (nu=0.1, 300 trees): test vs train R^2.
depths = [1, 2, 3, 5]
train_r2, test_r2 = [], []
for d in depths:
    gbr = GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=d, random_state=SEED
    ).fit(X_train, y_train)
    train_r2.append(gbr.score(X_train, y_train))
    test_r2.append(gbr.score(X_test, y_test))
    print(f"depth={d}: test R2 = {test_r2[-1]:.4f}, train R2 = {train_r2[-1]:.4f}")

fig = viz.plot_train_test_curve(
    depths, train_r2, test_r2, xlabel="max_depth (interaction order)", ylabel="R²"
)
ax = fig.axes[0]
ax.set_xticks(depths)
ax.set_title("depth = interaction order: stumps miss x₀·x₁, deep trees memorize")
plt.show()

**Read the figure.** Stumps (depth 1) stall at test R² ≈ 0.873 — and their training R² is only
≈ 0.905, so they under-fit even the training data: one-feature splits cannot represent x₀·x₁. Depth 2
jumps to ≈ 0.931 — the interaction is now within reach, and that is the single biggest gain. Depth 3 is
about the same on test (≈ 0.929) while its training R² keeps rising; by depth 5 the training R² is ≈ 0.998
but the test R² has slipped to ≈ 0.923 — the trees have started to memorize.

So gradient boosting wants **shallow** trees: deep enough for the real interaction order (here 2), not so
deep that each tree memorizes. scikit-learn's default depth is 3. This is the **mirror image** of a random
forest, which grows its trees **deep on purpose** (Chapter 06): boosting wants weak learners it can
correct slowly; bagging wants strong ones it can average.

## The three dials together

| dial | what it does | the danger | our Friedman reading |
|---|---|---|---|
| **ν** (learning rate) | size of each tree's step | too large → fast over-fitting | ν = 0.1 beat ν = 1, and was safer |
| **n_estimators** | number of steps | too many at large ν → over-fits; too few → under-fits | ν = 1 peaked at ~18; ν = 0.01 had not arrived by 1000 |
| **max_depth** | interaction order per tree | too deep → memorizes; too shallow → misses interactions | depth 2 captured x₀·x₁; depth 5 began to memorize |

ν and n_estimators trade off against each other — a **small ν with many trees** is the sweet spot, and it
generalizes best. Depth is set by the problem's real interaction order, kept as shallow as that allows.
The one danger unique to boosting is too many trees at a large ν. The principled cure — set a small ν and
a shallow depth, then **stop adding trees once a validation score stops improving** — is the subject of
Notebook 5.

## Your turn

1. **The turnaround.** From the ν × trees figure, name the rough number of trees at which ν = 1 reaches
   its best test R², and say in one sentence why every tree after that *hurts*.
2. **A new learning rate.** Re-run the ν-sweep with ν = 0.05. Does its best test R² (within 1000 trees)
   beat ν = 0.1's, and roughly how many trees does it need to get there? Relate your answer to the
   ν × n_estimators trade-off.
3. **Enough depth.** Friedman's only interaction is the two-feature x₀·x₁. Predict what `max_depth = 2`
   versus `max_depth = 1` should do to the test R², check it against the depth figure, then argue why no
   depth beyond 2 is *needed* to fit this target — even though 3 is the default.

## What you built

- You felt **ν as a step-size brake** and measured the **ν × n_estimators trade-off**: a smaller step
  needs more trees but reaches a better, safer place.
- You saw the headline that **more trees can make gradient boosting worse at a large ν** — the test curve
  bottoms then rises — and the mechanistic contrast with a random forest, whose test error never climbs
  with more trees.
- You learned that **tree depth is an interaction order**: shallow trees, matched to the problem (here,
  depth 2 for the x₀·x₁ interaction) — the mirror image of a random forest's deep trees.

**Vocabulary you now own:** learning rate / shrinkage ν · the ν × n_estimators trade-off · staged test
curve · over-fitting with too many trees · interaction order · weak (shallow) learner.

## Going further (optional)

- **Shrinkage as regularization.** Friedman's original paper recommends a small learning rate with
  correspondingly more trees; empirically ν ≤ 0.1 generalizes best (Friedman 2001, §5; ESL §10.12). The
  cost is compute — many small steps — which is part of why the fast histogram implementations of
  Chapters 9–10 matter.
- **Depth and interaction order.** Reading tree depth as the maximum interaction order a model can express
  is made precise in ESL §10.11; depth 2 ("two-way interactions") is a common sweet spot.
- **Two more regularizers, in Notebook 5.** Sampling a fraction of the rows per tree (`subsample`,
  Friedman 2002) is stochastic gradient boosting; and **early stopping** halts the rising test curve
  automatically. Both are the natural sequel to this notebook.
- **Same word, different mathematics.** A forest's test error is roughly monotone in the number of trees
  (Chapter 06, Notebook 4); a boosting ensemble's is U-shaped at a large ν. The number of trees means
  opposite things for the two methods.

## References

- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
  (§5, shrinkage.)
- Friedman, J. H. (1991). *Multivariate adaptive regression splines.* Annals of Statistics, 19(1), 1–67.
  DOI [10.1214/aos/1176347963](https://doi.org/10.1214/aos/1176347963). (The regression function used by
  `make_friedman1`; also a benchmark in Breiman 1996.)
- Friedman, J. H. (2002). *Stochastic gradient boosting.* Computational Statistics & Data Analysis,
  38(4), 367–378. DOI
  [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2). (`subsample`.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.11–10.12. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
  (Interaction order; shrinkage.)
- More trees never hurts a forest: Chapter 06, Notebook 4. How boosting overfits: Chapter 07, Notebook 3.

*Previous: 03 — Gradient boosting for classification.*  ·  *Next: 05 — The estimator and its parameters.*